<a href="https://colab.research.google.com/github/lutfikhamami/llm_engineering/blob/latihan/Week_3_day_4_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Models

Looking at the lower level API of Transformers - the models that wrap PyTorch code for the transformers themselves.

This notebook can run on a low-cost or free T4 runtime.


## One more reminder

**Pro-tip:**

In the middle of running a Colab, you might get an error like this:

> Runtime error: CUDA is required but not available for bitsandbytes. Please consider installing [...]

This is a super-misleading error message! Please don't try changing versions of packages...

This actually happens because Google has switched out your Colab runtime, perhaps because Google Colab was too busy. The solution is:

1. Kernel menu >> Disconnect and delete runtime
2. Reload the colab from fresh and Edit menu >> Clear All Outputs
3. Connect to a new T4 using the button at the top right
4. Select "View resources" from the menu on the top right to confirm you have a GPU
5. Rerun the cells in the colab, from the top down, starting with the pip installs

And all should work great - otherwise, ask me!

In [1]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [2]:
# library Transformers:
# AutoTokenizer = mengambil tokenizer yang cocok untuk model tertentu secara otomatis,(nama model, dia pilihkna tokenizer yang benar) ini yang mengubah teks jadi token ID dan sebaliknya
# AutoModelForCausalLM = memuat model generatif(causal/autoregressive LM) tipe model yang menebak token berikutnya
# TextStreamer = utilitas untuk streaming, menampilkan teks token-demi-token secara real time saat "model.generate" berjalan, alih alih menunggu semuanya selesai
# BitsAndBytesConfig = kelas konfigurasi untuk quantization, untuk set "load_in4bit","nf4","double_quant" dan "compute_dtype". kelas ini adalah jembatan ke library

# import gc = Garbage collector - module standar bawaan python, untuk membersihkan memory secara manual lewat "gc.collect()"

from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [3]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

### Accessing Llama

Yesterday you should have received approval to use this model:

https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

You can either use that today, or it's faster if you get approval for this model too.

https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

Select this link to see if you need to request approval too. Pick the version of Llama that you want below by commenting out one of these! Or skip Llama altogether.

In [4]:
# instruct models and 1 reasoning model

# Llama 3.1 is larger and you should already be approved
# see here: https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct

# LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# Llama 3.2 is smaller but you might need to request access again
# see here: https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct

LLAMA = "meta-llama/Llama-3.2-1B-Instruct"

PHI = "microsoft/Phi-4-mini-instruct"
GEMMA = "google/gemma-3-270m-it"
QWEN = "Qwen/Qwen3-4B-Instruct-2507"
DEEPSEEK = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

In [5]:
messages = [
    {"role": "system", "content": "tell a joke for a room of Data Scientists"}
]

# Accessing Llama 3.1 from Meta

In order to use the fantastic Llama 3.1, Meta does require you to sign their terms of service.

Visit their model instructions page in Hugging Face:
https://huggingface.co/meta-llama/Meta-Llama-3.1-8B

At the top of the page are instructions on how to agree to their terms. If possible, you should use the same email as your huggingface account.

In my experience approval comes in a couple of minutes. Once you've been approved for any 3.1 model, it applies to the whole family of models.

If you have any problems accessing Llama, please see this colab, including some suggestions if you don't get approved by Meta for any reason.

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8

In [6]:
# Quantization Config - this allows us to load the model into memory and use less memory
# Fungsi cell ini: membuat satu "resep" pengaturan quantization lalu menyimpannya ke variabel "quant_config" - supaya nanti bisa di serahkan ke model saat di jalankan


quant_config = BitsAndBytesConfig(
    load_in_4bit=True, # muat model ini dalam 4 bit, menurunkan presisi tiap parameter dari 16bit jadi 4bit sehingga hemat memory 4x.
    bnb_4bit_use_double_quant=True, # hemat ekstra ? (ya, kompres faktor skala)
    bnb_4bit_compute_dtype=torch.bfloat16, # memisahkan penyimpanan dari perhitungan, parameter di simpan dalam 4bit. "brain float point 16-bit" format angka 16bit
    bnb_4bit_quant_type="nf4" # "NormalFloat-4bit" format angka 4bit - jauh lebih kecil dari bfloat16
)

If the next cell gives you a 403 permissions error, then please check:
1. Are you logged in to HuggingFace? Try running `login()` to check your key works
2. Did you set up your API key with full read and write permissions?
3. If you visit the Llama3.1 page at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B, does it show that you have access to the model near the top?

And work through my Llama troubleshooting colab:

https://colab.research.google.com/drive/1deJO03YZTXUwcq2vzxWbiBhrRuI29Vo8


In [7]:
# Tokenizer
# cell ini tugasnya menyiapkan input - mengubah pesan(teks) jadi token ID yang ready di masukan ke dalam model di GPU

tokenizer = AutoTokenizer.from_pretrained(LLAMA) # mengunduh/memuat tokenizer yang cocok untuk model Llama(LLAMA), kenapa "Auto"? karena tiap model punya tokenizer sendiri dengan aturan berbeda (kosakata, spesial token, cara memecah kata dll) dengan "AutoTokenizer" tidak perlu tau kelas tokenizer spesifik, cukup kasih nama model, lalu HF otomatis memilih tokenizer yang benar
tokenizer.pad_token = tokenizer.eos_token # Eos_token(end-of-sentence) token khusus penanda "kalimat selesai" tiap tokenizer punya ini. Pad_token(padding) token khusus untuk mengisi ruang kosong, digunakan saat memproses beberapa input sekaligus yang panjangnya beda
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda") # mengubah pesan jadi token di GPU

In [8]:
inputs

# output: hasil tokenizer yang mengubah pesan jadi angka, isinya sebuah dictionary dengan 2 key (input_ids, attention_mask)
# input_ids : deretan angka yang mewakili teks, tiap angka adalah satu token dari kosakata Llama, angka besar di atas 128000 itu spesail token(penanda struktur) angka lainnya kata/potongan kata biasa
# input_ids bukan cuma pesan mentah - sudah di bungkus lengkap dengan struktur chat(header system, header user, penanda selesai) hasil "apply_chat_template", wujud "teks yang sudah jadi token ID"
# attention_mask: token mana yang harus di perhatikan, ini deretan angka 1 dan 0, 35 angka 1, artinya semua token nyata, tidak ada tambalan

tensor([[128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
             25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
            220,   2705,   5033,    220,   2366,     21,    271,  73457,    264,
          22380,    369,    264,   3130,    315,   2956,  57116, 128009]],
       device='cuda:0')

In [9]:
# The model
# Memuat model
# AutoModelForCausalLM.from_pretrained(LLAMA): mengunduh dan membangun model Llama khusus untuk causal LM - tipe model generatif yang menebak token berikutnya
# device_map="auto": memberi tahu HF (lewat library accelerate)
# quantization_config=quant_config: resep dari variabel quant_config di jalankan
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [10]:
# Mengecek jejak memory
# 1e6 : adalah cara python menulis 1.000.000(1juta), membagi byte dengan sejuta mengubah satuan jadi megabyte(MB) cth: (misal 1050279936) di bagi sejuta jadi ~1050MB (enak di lihat)


memory = model.get_memory_footprint() / 1e6 # method bawaan yang mengembalikan total memory yang di pakai model dalam satuan byte
print(f"Memory footprint: {memory:,.1f} MB") # (,)menambah pemisah ribuan(misal 1,050). (.1f) membulatkan ke 1 angka di belakang koma (misal 1050.3)

Memory footprint: 1,012.0 MB


## Looking under the hood at the Transformer model

The next cell prints the HuggingFace `model` object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my [YouTube intro playlist](https://www.youtube.com/playlist?list=PLWHe-9GP9SMMdl6SLaovUQF2abiLGbMjs) for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

- It consists of layers
- There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
- There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
- There is an LM Head layer at the end; this produces the output

Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, I've asked our mutual friend to take this printout and make a tutorial to walk through each layer. This also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [11]:
# Execute this cell and look at what gets printed; investigate the layers

model

# output: adalah blueprint lengkap aksitektur model Llama 3.21B, mencetak truktur sebagai pohon bertingkat,
# LlamaForCausalLM adalah object terluar, membunkus dua hal: (model) isi utama dan (lm_head) yang menghasilkan probabilitas token berikutnya
# Embeding = (embed_tokens): Embedding(128256, 2048): lapisan embeding mengubah token jadi vektor Angka 128256 = jumlah total token. 2048= dimensi vektornya
# linear: tiap linear (termasuk linear4bit) adalah satu lapisan transformasi: menerima sekumpulan angka masuk lalu menghasilkan kumpulan angka keluar dua parameter itu mendefinisikan ukurannya:
# in_features: berapa banyak angka yang masuk ke layer ini (dimensi data, bukan jumlah token)
# out_features: berapa banyak angka yang keluar dari layer ini (dimensi data, bukan jumlah token)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear4bit(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm

### And if you want to go even deeper into Transformers

In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:  
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:  
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not neceesary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [14]:
# OK, with that, now let's run the model!

outputs = model.generate(inputs, max_new_tokens=80)
outputs[0]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


tensor([128000, 128006,   9125, 128007,    271,  38766,   1303,  33025,   2696,
            25,   6790,    220,   2366,     18,    198,  15724,   2696,     25,
           220,   2705,   5033,    220,   2366,     21,    271,  73457,    264,
         22380,    369,    264,   3130,    315,   2956,  57116, 128009, 128006,
         78191, 128007,    271,   8586,    596,    264,  22380,  41891,    311,
           264,   3130,    315,    828,  14248,   1473,  10445,   1550,    279,
         10550,    733,    311,  15419,   1980,  18433,    433,    574,   8430,
           264,   2697,    330,    695,     12,   2230,      1,    323,   4460,
           311,    330,   4734,      1,   1202,  21958,   2268,   2028,  22380,
         11335,    389,    279,  11156,   3878,   1511,    304,    828,   8198,
            11,   1778,    439,    330,    695,     12,   2230,      1,    320,
         57865,    828,    374,  46946,    477,  50500,      8,    323,    330,
          4734,      1,    320,  57865, 

In [15]:
tokenizer.decode(outputs[0])

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 06 Aug 2026\n\ntell a joke for a room of Data Scientists<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\nHere\'s a joke tailored to a room of data scientists:\n\nWhy did the dataset go to therapy?\n\nBecause it was feeling a little "data-iled" and needed to "process" its emotions!\n\nThis joke plays on the technical terms used in data science, such as "data-iled" (meaning data is messy or corrupted) and "process" (meaning to analyze or'